# Creating EBRAINS metadata with fairgraph

[fairgraph](https://fairgraph.readthedocs.io) represents the metadata in the
[EBRAINS Knowledge Graph](https://docs.kg.ebrains.eu) (KG) as Python objects. The companion tutorial
for data *consumers* uses it to find and download metadata. This one is about the other direction:
**creating and updating metadata of your own**.

> **Who this is for.** Most data providers don't work directly with fairgraph: they describe their
> dataset in the **Metadata Wizard**, a web application that asks the questions and builds the
> metadata for them. However, if you have many datasets to register, metadata that already exists 
> in some other form, an automated pipeline, any other complex use case, or just want to understand
> what goes on under the hood, this tutorial is for you.

> **Everything in this notebook writes to the pre-production Knowledge Graph**
> (`core.kg-ppd.ebrains.eu`). It is a clone of the production KG, updated on a regular basis.
> Nothing you do here reaches the production KG, and any changes you make will disappear
> next time the clone is updated. This makes it ideal for experimentation.
> The code is identical for production; only the host changes.

### How to use this notebook

Sections marked **&#11088;** are the core, and we will work through them together.
Sections marked **&#9711;** are for afterwards.

1. &#11088; Setup, and a recap
2. &#11088; Build a node, and validate it
3. &#11088; Save it
4. &#11088; Create, or update?
5. &#11088; A complete dataset
6. &#11088; Working with others: collab spaces
7. &#11088; What happens next: the curation workflow
8. &#9711; Collections, for datasets of real size
9. &#9711; Registering files by hand, and why you usually should not
10. &#11088; Cleaning up after yourself
11. &#9711; Where to go next

**Please run section 10 before you leave**, so you do not leave test records behind.

## &#11088; 1. Setup, and a recap

If you are in the [EBRAINS Lab](https://lab.ebrains.eu) with a recent kernel from the EBRAINS
Software Distribution, fairgraph is already installed; otherwise `pip install fairgraph`.

Authentication works exactly as for reading: in the Lab your token is picked up automatically,
and elsewhere the client prints a log-in URL, or takes a token from `KG_AUTH_TOKEN` or from
`KGClient(token=...)`. Use of the KG API means accepting the
[Terms of Use](https://docs.kg.ebrains.eu/search-terms-of-use.html).

In [1]:
# Everything this notebook hard-codes, in one place.

KG_HOST = "core.kg-ppd.ebrains.eu"   # PRE-PRODUCTION. Practise here, not on the production KG.
SPACE = "myspace"                    # your own private space; nobody else can see it

# An unmistakably fictional dataset (at least for a neuroscience database), so that nothing here looks like real data.
DATASET_NAME = "Comedic timing in silent film"
DATASET_SHORT_NAME = "LH-001"

In [2]:
from datetime import date
from uuid import uuid4

import fairgraph
from fairgraph import KGClient, IRI, Collection
from fairgraph.utility import as_list
import fairgraph.openminds.core as omcore
import fairgraph.openminds.controlled_terms as terms

print("fairgraph version:", fairgraph.__version__)

client = KGClient(host=KG_HOST)
user = client.user_info()
print(f"connected to {KG_HOST} as {user.given_name} {user.family_name}")

fairgraph version: 0.13.3
connected to core.kg-ppd.ebrains.eu as Testing Davison


Note that we have **not** turned error handling off. The "data consumer" notebook silences it, because
public records you did not write are not yours to fix. Now that you are the one creating metadata,
you want to hear about problems.

In [3]:
# Where may you write? (We print only the answer, not the list - space names can be private.)
writable = client.spaces(names_only=True, permissions=["WRITE"])
print(f"you can write to {len(writable)} space(s)")
print(f"'{SPACE}' is one of them: {SPACE in writable}")

you can write to 9 space(s)
'myspace' is one of them: True


### A one-minute recap, if you missed the "data consumer" tutorial

The KG is a set of metadata **nodes** joined by **typed links**. Each openMINDS type is a Python
class, and each property is an attribute. Let's look at an existing record.

In [4]:
existing = omcore.DatasetVersion.list(client, size=1)

if existing:
    existing[0].show()
else:
    print("no released dataset versions on this server - that is fine, we are about to make one")

id                       https://kg.ebrains.eu/api/instances/005ae509-b575-47f6-b4e1-2c9174ed80ce
space                    dataset
type                     https://openminds.om-i.org/types/DatasetVersion
accessibility            KGProxy([ProductAccessibility], id="b2ff7a47-b349-48d7-8ce4-cf51868675f1")
authors                  [KGProxy([Consortium, Organization, Person], id="8a38f341-5b88-4cb2-b8b7-6473a676d892"), KGPr ...
data_types               [KGProxy([SemanticDataType], id="2a07e0f3-f48a-48f9-9fe3-9164909a8408"), KGProxy([SemanticDat ...
digital_identifier       KGProxy([DOI, IdentifiersDotOrgID], id="5e6ad949-084d-40c7-a2c8-b04bdcea824b")
ethics_assessment        KGProxy([EthicsAssessment], id="5837d74e-c3e3-4ddd-8fe9-6d0eea6a0d14")
experimental_approaches  [KGProxy([ExperimentalApproach], id="2ba8de18-a092-4e5e-82af-394cf32fe399"), KGProxy([Experim ...
full_name
homepage                 https://www.fz-juelich.de/inm/inm-1/EN/Home/home_node.html
how_to_cite              **For re

Three things to carry forward:

1. values like `KGProxy([License], ...)` are **links that have not been followed**. Links are never
   followed automatically; you ask for them with `.resolve(client)` or `follow_links={...}`.
2. openMINDS splits a research product in two: `Dataset` (the thing) and `DatasetVersion` (this
   release of it). The version holds the files and the release date; the parent holds information that 
   is common to all versions, for example the name and description. You will create both in section 5.
3. `cls.property_names` and `help(cls)` tell you what a class can hold.

## &#11088; 2. Build a node, and validate it

Creating metadata is ordinary Python: instantiate a class, pass properties as arguments. Nothing
touches the network until you call `save()`.

Let's register two people and their employer. We will get it wrong first, on purpose, because the
mistake below is an easy one to make.

In [5]:
mgm = omcore.Organization(full_name="Metro-Goldwyn-Mayer", short_name="MGM")

stan = omcore.Person(given_name="Stan", family_name="Laurel", affiliations=mgm)
stan.show()

id            None
space         None
type          https://openminds.om-i.org/types/Person
affiliations  Organization(full_name='Metro-Goldwyn-Mayer', short_name='MGM', space=None, id=None)
family_name   Laurel
given_name    Stan


That looks reasonable, but it is wrong. `validate()` checks a node against the openMINDS schema,
locally, with no network and no account needed. It returns a dict of problems, empty if all is well.

In [6]:
stan.validate()

{'type': ['affiliations: Expected Affiliation, value contains Organization']}

`affiliations` does not hold an `Organization`; it holds an `Affiliation`, which is a small node
recording *that* someone was a member of an organization, and when.

When you are unsure what a property wants, ask it:

In [7]:
print("affiliations wants:", [t.__name__ for t in omcore.Person.get_property("affiliations").types])
print("Affiliation has   :", omcore.Affiliation.property_names)
print("start_date wants  :", omcore.Affiliation.get_property("start_date").types)

affiliations wants: ['Affiliation']
Affiliation has   : ['end_date', 'member_of', 'start_date']
start_date wants  : (<class 'datetime.date'>,)


In [8]:
stan.affiliations = [omcore.Affiliation(member_of=mgm, start_date="23rd February 1942")]
stan.validate()

{'type': ['start_date: Expected date, value contains str']}

Closer, but `start_date` wants a `datetime.date`, not a string. openMINDS is strict about types
precisely so that everyone's dates mean the same thing.

In [9]:
stan.affiliations[0].start_date = date(1942, 2, 23)
print(stan.validate())

ollie = omcore.Person(
    given_name="Oliver",
    family_name="Hardy",
    alternate_names=["Ollie"],
    affiliations=omcore.Affiliation(member_of=mgm, start_date=date(1942, 2, 23)),
)
print(ollie.validate())

{}
{}


Two empty dicts: both nodes are well-formed.

**Validate before every save.** It costs nothing, it runs offline, and the messages are, hopefully, informative.

> **Try it yourself.** `omcore.Organization` can hold a persistent identifier such as a ROR ID.
> What is the property called, what type does it want, and what does `validate()` say if you pass a
> plain string instead?

## &#11088; 3. Save it

`save()` sends the node to the KG. The first time, you say which space it belongs in.

In [10]:
created = []       # we will collect what we make, so section 10 can clean it up

stan.save(client, space=SPACE)
ollie.save(client, space=SPACE)

created += [stan, ollie, mgm]

print("Stan :", stan.uuid)
print("Ollie:", ollie.uuid)
print("MGM  :", mgm.uuid)

Stan : 23fe4077-dc88-47f2-be53-5f589ed3f6f8
Ollie: 6612dda2-ae47-4ac3-9dd2-0e2ec5160bb4
MGM  : afa4d7b4-8084-4511-8444-1ea4b5c7cfeb


`mgm` has an id too, although we never saved it explicitly: by default `save()` is **recursive** and
stores the child nodes a node links to. More on that at the end of section 4.

Now read them back. Note `release_status="in progress"`: without it you would see nothing.

In [11]:
# The default release_status is "released" - which your brand-new work is not.
print("found with the default release_status:",
      len(omcore.Person.list(client, space=SPACE, family_name="Laurel")))
print("found with release_status='in progress':",
      len(omcore.Person.list(client, space=SPACE, release_status="in progress",
                             family_name="Laurel")))

for person in omcore.Person.list(client, space=SPACE, release_status="in progress",
                                 family_name="Laurel"):
    print(f"- {person.full_name} ({person.uuid})")

found with the default release_status: 0
found with release_status='in progress': 1


- Stan Laurel (23fe4077-dc88-47f2-be53-5f589ed3f6f8)


This is the single most common confusion for new providers: **everything you save is "in progress"**,
and the default everywhere in fairgraph is `release_status="released"`. Your own work is invisible
until you ask for it by name. Section 7 explains what "released" means and who can grant it.

## &#11088; 4. Create, or update?

When you call `save()` on an object that has no id, fairgraph has to decide whether you mean
"create a new node" or "update one that already exists". It decides by querying the KG for a node
matching a *subset* of the properties, and each class declares which subset.

In [12]:
for cls in (omcore.Person, omcore.Organization, omcore.Dataset, omcore.DatasetVersion, omcore.File):
    print(f"{cls.__name__:<16} {cls.existence_query_properties}")

Person           ('given_name', 'family_name')
Organization     ('full_name',)
Dataset          ('short_name',)
DatasetVersion   ('short_name', 'version_identifier')
File             ('iri', 'hashes')


So two `Person` objects with the same given and family name are considered the same person, however
much else differs. Let's prove it: a brand-new object, built from scratch, with no id.

In [13]:
another_stan = omcore.Person(given_name="Stan", family_name="Laurel")
another_stan.save(client, space=SPACE)

print("original:", stan.uuid)
print("new one :", another_stan.uuid)
print("same node:", stan.uuid == another_stan.uuid)

original: 23fe4077-dc88-47f2-be53-5f589ed3f6f8
new one : 23fe4077-dc88-47f2-be53-5f589ed3f6f8
same node: True


One node, not two. That is usually what you want; it is what stops a hundred dataset records
creating a hundred duplicate authors.

When you genuinely need a second node with the same identifying properties, generate the id
yourself and fairgraph will not go looking:

In [14]:
namesake = omcore.Person(
    given_name="Stan",
    family_name="Laurel",
    id=client.uri_from_uuid(str(uuid4())),
)
namesake.save(client, space=SPACE)

print("original:", stan.uuid)
print("namesake:", namesake.uuid)
print("distinct:", namesake.uuid != stan.uuid)

original: 23fe4077-dc88-47f2-be53-5f589ed3f6f8
namesake: 0c6a3b24-02c5-4af8-994a-d74997299d72
distinct: True


### Updating

To change a node, edit the attribute and save again. You do not pass `space`, fairgraph already
knows where the node lives.

In [15]:
stan.alternate_names = ["Arthur Stanley Jefferson"]
stan.save(client)

refetched = omcore.Person.from_id(stan.uuid, client, release_status="in progress")
print(refetched.alternate_names)

None


### The trap: matching is "contains", not "equals"

Here is the part to remember. In released versions of fairgraph the existence query matches a string
property when the stored value merely **contains** the one you are saving. `Dataset` is matched on
`short_name` alone, so a dataset called "LH-001-extended" will be found when you save one called
"LH-001", and `save()` would then *overwrite* it.

We can demonstrate this without destroying anything, because `exists()` performs the same query that
`save()` uses, but changes nothing.

In [16]:
# A tag unique to this run, so the demonstration works even if you run the notebook twice.
tag = f"LH-{uuid4().hex[:6]}"

decoy = omcore.Dataset(
    full_name=f"{DATASET_NAME} (extended cut)",
    short_name=f"{tag}-extended",
    description="A deliberately similar record, to demonstrate existence matching.",
)
decoy.save(client, space=SPACE)
created.append(decoy)
print("saved:", decoy.short_name, decoy.uuid)

saved: LH-7c41f9-extended 267fb249-d1da-49bc-9e0c-12499fea818d


In [17]:
# a *different* dataset, whose short name happens to be a prefix of the one above
probe = omcore.Dataset(full_name=DATASET_NAME, short_name=tag)

print(f"saved short_name: {tag}-extended")
print(f"probe short_name: {tag}")
print()

if probe.exists(client):
    print("The KG says this already exists, and points at", probe.uuid)
    print("which is the record we just saved:", probe.uuid == decoy.uuid)
    print()
    print("Calling save() instead of exists() would have OVERWRITTEN that record")
    print("with this one. That is the trap.")
else:
    print("No match: this version of fairgraph matches short names exactly.")
    print("Exact matching is in development; in the released versions (0.13.3, 0.15.0)")
    print("this same code DOES match, and save() would have overwritten the record above.")

saved short_name: LH-7c41f9-extended
probe short_name: LH-7c41f9



The KG says this already exists, and points at 267fb249-d1da-49bc-9e0c-12499fea818d
which is the record we just saved: True

Calling save() instead of exists() would have OVERWRITTEN that record
with this one. That is the trap.


Had we called `save()` there, we would have overwritten the decoy with a different dataset.

Exact matching, and an `existence_match` option to choose between the two behaviours, will be in the future fairgraph v0.16.0; 
the published documentation already describes them, but they are **not in a released version**. 
The cell above reports which behaviour *your* installation has. Until exact matching is
released:

- be careful with short names that are prefixes of one another;
- check with `exists()` before saving when you are not sure;
- and generate ids yourself when you know a node is new.

### Recursive saving

`save()` stores child nodes by default. For a handful of objects that is convenient. For a pipeline
it is better to be explicit: save the children yourself, then save the parent with
`recursive=False`, so that you know exactly what was written and when.

> **Try it yourself.** What are `DatasetVersion.existence_query_properties`? Given those, what would
> happen if you saved two different versions of a dataset that both had `version_identifier="v1"`
> but different short names?

## &#11088; 5. A complete dataset

Now the real thing: a `Dataset`, a `DatasetVersion`, the files, and all the metadata openMINDS
requires. Start naively and let `validate()` tell us what is missing.

In [18]:
dsv = omcore.DatasetVersion(
    full_name=DATASET_NAME,
    short_name=DATASET_SHORT_NAME,
    version_identifier="v1.0",
)
dsv.validate()

{'required': ['accessibility is required, but was not provided',
  'data_types is required, but was not provided',
  'digital_identifier is required, but was not provided',
  'ethics_assessment is required, but was not provided',
  'experimental_approaches is required, but was not provided',
  'full_documentation is required, but was not provided',
  'license is required, but was not provided',
  'release_date is required, but was not provided',
  'techniques is required, but was not provided',
  'version_innovation is required, but was not provided']}

Twelve required properties. That is openMINDS being demanding on purpose: every one of them is
something that someone looking at your dataset needs in order to understand what it is and whether they may use it.

Several are links to **controlled terms**, and those you do not have to look up at all. openMINDS
ships its vocabularies with the library, so `by_name()` **without** a client returns the term
directly, with no network request. The id you get is a semantic IRI such as
`https://openminds.om-i.org/instances/productAccessibility/freeAccess`. The Knowledge Graph
keeps a dual identity for these nodes, the semantic IRI alongside the usual UUID-based one, so a
link made this way resolves perfectly well.

Passing `client` would also work, but it costs a network request per term.


In [19]:
# No client: these come from the openMINDS instance library that ships with the package.
controlled = {
    "accessibility": terms.ProductAccessibility.by_name("free access"),
    "data_types": terms.SemanticDataType.by_name("raw data"),
    "ethics_assessment": terms.EthicsAssessment.by_name("EU compliant"),
    "experimental_approaches": terms.ExperimentalApproach.by_name("electrophysiology"),
    "techniques": terms.Technique.by_name("whole cell patch clamp"),
    "license": omcore.License.by_name("CC-BY-4.0"),
}

for property_name, term in controlled.items():
    print(f"{property_name:<26} {term.id}")


accessibility              https://openminds.om-i.org/instances/productAccessibility/freeAccess
data_types                 https://openminds.om-i.org/instances/semanticDataType/rawData
ethics_assessment          https://openminds.om-i.org/instances/ethicsAssessment/EUCompliant
experimental_approaches    https://openminds.om-i.org/instances/experimentalApproach/electrophysiology
techniques                 https://openminds.om-i.org/instances/technique/wholeCellPatchClamp
license                    https://openminds.om-i.org/instances/licenses/CC-BY-4.0


Each of those terms is also available as a **class attribute**, which is often tidier:

```python
terms.UnitOfMeasurement.byte                  # same object as by_name("byte")
terms.ExperimentalApproach.electrophysiology
terms.SemanticDataType.raw_data
```

The attribute name is the term's name with spaces and punctuation flattened, so the two forms suit
different cases. For a short name the attribute is shorter, and your editor will complete it. For
anything with punctuation the mangling is a guess (`License.cc_by_4_0`,
`ContentType.text_tab_separated_values`) and `by_name("CC-BY-4.0")` is both clearer and written the
way the term actually is. They return the same object; use whichever makes the line easier to read.

This matters again in section 8: when you upload a `Collection`, fairgraph deliberately skips any
node whose id begins with `https://openminds.om-i.org/instances` since they already exist in the Knowledge Graph.

The remaining required properties are not controlled terms, so we supply them ourselves: a link to
documentation, a release date, and a sentence saying what is new in this version.

One of them we deliberately leave out, let's see what `validate()` makes of that.


In [20]:
dsv = omcore.DatasetVersion(
    full_name=DATASET_NAME,
    short_name=DATASET_SHORT_NAME,
    version_identifier="v1.0",
    version_innovation="First release.",
    release_date=date(1942, 2, 23),
    full_documentation=omcore.WebResource(iri=IRI("https://example.org/LH-001/README.md")),
    **controlled,
)

print(dsv.validate())

{'required': ['digital_identifier is required, but was not provided']}


Not empty: `digital_identifier is required, but was not provided`. And there is nothing you can do
about it.

A dataset's DOI is not something a provider invents. EBRAINS mints it through
[DataCite](https://datacite.org) when the dataset is published, as part of curation, so at the point
where you are preparing metadata the DOI does not exist yet. The schema is not wrong to require it,
a *published* dataset must have one, but you cannot satisfy it, and you should not invent a
placeholder, because a DOI that does not resolve is worse than an absent one.

So this is the second thing to understand about `validate()`, after section 2: **a clean validation
is not a precondition for saving.** "Required" describes a complete, published record. Some of those
requirements are yours to meet, and some are filled in later by curators.

Now, where the data actually live.

A `FileRepository` records the location of the data (in practice a Collaboratory bucket) and who
hosts it. Note what we are *not* about to do: create a `File` node for each file. For datasets, the Knowledge
Graph has a service that ingests the files found in the bucket a `FileRepository` points at and
creates those nodes for you. As a provider you describe the repository; the file-level metadata
follows from the data themselves.

In [21]:
# Look the host up rather than constructing it: what comes back is the KG's own record, with its
# full registered name and its id. See the note below for why that matters.
ebrains = omcore.Organization.by_name("EBRAINS", client, match="contains")
print(f"{ebrains.full_name}  ({ebrains.uuid})")

repository = omcore.FileRepository(
    name=f"{DATASET_SHORT_NAME} repository",
    iri=IRI("https://data-proxy.ebrains.eu/api/v1/buckets/example-tutorial-bucket"),
    hosted_by=ebrains,
)

dsv.repository = repository
print(repository.validate())

EBRAINS Association Internationale Sans But Lucratif  (7dfdd91f-3d05-424a-80bd-6d1d5dc11cd3)
{}


`hosted_by` is **looked up**, not constructed. `by_name()` with `match="contains"` finds the record
the Knowledge Graph already holds (whose registered name is "EBRAINS Association Internationale
Sans But Lucratif") and what comes back carries that name and its id, so linking to it is all we
need to do. It is already saved; we neither write it nor delete it, which is why it is absent from
the save list and from the cleanup in section 10.

The habit to take away: when you need an organization, a person or a funding body, **look up what
the Knowledge Graph already holds and link to it**, rather than constructing a node from whatever
you happen to call it and relying on the existence query to reconcile the two.

Section 9 shows how to create `File` nodes by hand, for the cases where you have to, and why it is
the exception rather than the rule.

Finally the parent `Dataset`, which carries the authors and the description, and points at its
versions.

In [22]:
dataset = omcore.Dataset(
    full_name=DATASET_NAME,
    short_name=DATASET_SHORT_NAME,
    description=(
        "A fictional dataset, created during an EBRAINS tutorial, to demonstrate how metadata "
        "is registered with fairgraph. It contains no real data."
    ),
    authors=[stan, ollie],
    has_versions=[dsv],
)

print(dataset.validate())

{'required': ['digital_identifier is required, but was not provided']}


In [23]:
# One call saves the whole tree: recursive is the default.
dataset.save(client, space=SPACE, recursive=True)

created += [dataset, dsv, repository, dsv.full_documentation]

print("dataset        ", dataset.uuid)
print("dataset version", dsv.uuid)
print("repository     ", repository.uuid)
print("documentation  ", dsv.full_documentation.uuid)


dataset         9f298964-68b0-4062-a0b1-39084a55778c
dataset version b4e1a6ef-5083-49af-a76d-1a791aa09eeb
repository      3c655ca1-598c-4198-a538-dfd928115fa4
documentation   eda1ba40-04c2-49ac-aa71-aa799791cc25


That one call did rather a lot, and it is worth knowing what, because the instinct to save
everything by hand is strong.

`save()` is recursive by default: it walks the links, saves children before their parents so that
every link points at something that already has an id, and then saves the object you asked for. The
`WebResource`, the `FileRepository` and the `DatasetVersion` were all created on the way to saving
the `Dataset`.

Two things it did *not* do:

- it did not write the **controlled terms**. They are openMINDS library instances, already in the
  Knowledge Graph under their semantic IRIs, and nothing needed saying about them;
- it did not write the **hosting organization**. We looked that up, so the local object matches the
  stored one exactly, nothing is modified, and `save()` has nothing to do.

The alternative is `recursive=False`, where you save each node yourself, children first. It is more
code and easier to get wrong. What it buys you is knowing
exactly what was written and when, which matters in a pipeline that runs unattended. For a dataset
you are assembling interactively, the default is the right choice.

> **Try it yourself.** Add a second `DatasetVersion`, "v1.1", to the same `Dataset`, with
> `is_new_version_of` pointing at the first. What does `Dataset.has_versions` contain afterwards?


## &#11088; 6. Working with others: collab spaces

So far everything has gone into `myspace`, which is exactly as private as it sounds: **nobody else
can read it, and that includes EBRAINS curators.** It is a fine scratchpad, and a dead end for
anything you want published.

Shared work lives in a **collab space**: a KG space attached to a Collab in the
[EBRAINS Collaboratory](https://wiki.ebrains.eu), whose name begins with `collab-` and whose access
follows the Collab's membership. You create the Collab in the Collaboratory, decide who belongs to
it, and ask for a KG space for it.

In [24]:
writable = client.spaces(names_only=True, permissions=["WRITE"])
collab_spaces = [name for name in writable if name.startswith("collab-")]

print(f"you can write to {len(writable)} space(s)")
print(f"  'myspace'      : {'myspace' in writable}")
print(f"  collab spaces  : {len(collab_spaces)}")

if not collab_spaces:
    print("\nYou have no collab space yet; the code below is what you would run once you do.")

you can write to 9 space(s)
  'myspace'      : True
  collab spaces  : 8


A new space has to be **configured** before you can store anything in it: you declare which types
you intend to put there.

```python
types = [omcore.Dataset, omcore.DatasetVersion, omcore.Person,
         omcore.Organization, omcore.FileRepository, omcore.WebResource]
client.configure_space("collab-my-project", types)
```

After that it behaves like any other space: `dataset.save(client, space="collab-my-project")`, and
`client.space_info("collab-my-project", release_status="in progress")` to see what is in it.

### Moving what you have already made

Drafting in `myspace` and moving the result across is a reasonable way to work. `move_to_space()`
does that, one instance at a time:

```python
client.move_to_space(dataset.id, "collab-my-project")
```

**It is not recursive.** The dataset moves; its version, repository, authors and documentation stay
where they are, and you end up with links that cross spaces — readable only by someone who can read
both. Move the children too, and `children()` is how you find them:

```python
for child in dataset.children(client, follow_links={"has_versions": {}}):
    client.move_to_space(child.id, "collab-my-project")
```

Saving into the right space from the start avoids the whole exercise, which is the better habit once
you know where the metadata belong.

## &#11088; 7. What happens next: the curation workflow

You have metadata in the KG. It is **not** public, and it will not appear in the
[Search UI](https://search.kg.ebrains.eu). Let's see exactly where it stands.

In [25]:
# space_info() reports every type in the space. We show only the types this notebook creates -
# your myspace may well contain other things - and note that one key is a plain string, not a class.
ours = {omcore.Dataset, omcore.DatasetVersion, omcore.Person, omcore.Organization,
        omcore.File, omcore.FileRepository, omcore.DOI, omcore.WebResource}

info = client.space_info(SPACE, release_status="in progress")
for cls, count in sorted(info.items(), key=lambda item: str(getattr(item[0], "__name__", item[0]))):
    if count and cls in ours:
        print(f"{cls.__name__:<22} {count}")

DOI                    1
Dataset                2
DatasetVersion         2
File                   6
FileRepository         2
Organization           3
Person                 6
WebResource            1


Every node is "in progress". Making metadata "released" (citable, visible in the Search UI,
retrievable without `release_status="in progress"`) is a **curation step carried out by EBRAINS**,
not something a data provider does.

The real pipeline, once your metadata are ready:

1. create a Collab in the [EBRAINS Collaboratory](https://wiki.ebrains.eu) and get a KG space for
   it, as in section 6. **Curators cannot see `myspace`**, so metadata that stays there can never
   be curated;
2. prepare the metadata in that collab space, with fairgraph, as we have just done;
3. upload the data themselves to a Collaboratory bucket, and point `FileRepository.iri` at it.
   The files in the bucket are ingested into the KG as `File` nodes by a service, so you do not
   register them one by one;
4. ask EBRAINS to curate: **support@ebrains.eu**, or through the data curation request form. Give them the
   name of the collab space;
5. curators review, ask questions, correct, and release;
6. EBRAINS mints a DOI through DataCite, and the dataset appears in the Search UI.

That is the route when you are producing the metadata yourself. Most data providers never do: they
describe their dataset in the Metadata Wizard, and it reaches the Knowledge Graph by a different
path, with curators doing the work in between. Either way, what finally lands in the KG looks like
what you have just built.

## &#9711; 8. Collections, for datasets of real size

Saving node by node is fine for a dozen. For a dataset with a thousand files you want a
`Collection`: build the whole graph offline, write it to a single JSON-LD file you can read, review
and commit to version control, and upload it in one call.

In [26]:
# A small, self-contained set of new nodes.
troupe = omcore.Organization(full_name="Hal Roach Studios")
extras = [
    omcore.Person(given_name="Charley", family_name="Chase",
                  affiliations=omcore.Affiliation(member_of=troupe)),
    omcore.Person(given_name="Thelma", family_name="Todd",
                  affiliations=omcore.Affiliation(member_of=troupe)),
]

collection = Collection(*extras)
print(collection.statistics())

Counter({'Person': 2, 'Organization': 1})


In [27]:
collection.save("people_draft.jsonld")

import json as _json
with open("people_draft.jsonld") as fp:
    draft = _json.load(fp)
print(f"{len(draft['@graph'])} nodes written to people_draft.jsonld")
print(_json.dumps(draft["@graph"][0], indent=2)[:400], "...")

3 nodes written to people_draft.jsonld
{
  "@id": "_:000000",
  "@type": "https://openminds.om-i.org/types/Person",
  "affiliation": [
    {
      "@type": "https://openminds.om-i.org/types/Affiliation",
      "memberOf": {
        "@id": "_:000001"
      }
    }
  ],
  "familyName": "Chase",
  "givenName": "Charley"
} ...


That file is the artefact to review before anything reaches the KG: diff it, comment on it, put it
in a repository beside the data.

Uploading it is one call. `default_space` says where the nodes go, and `space_map` can route
different types to different spaces.

In [28]:
import os

# upload() keeps a log so that an interrupted upload can resume. That also means a *successful*
# upload will be skipped entirely on a re-run, so remove the log when you mean to start over.
if os.path.exists(".kg_upload_log.txt"):
    os.remove(".kg_upload_log.txt")

reloaded = Collection()
reloaded.load("people_draft.jsonld")
reloaded.upload(client, default_space=SPACE, verbosity=2)

# upload() assigns KG ids to the nodes in *this* collection, so these are the objects to clean up -
# the `extras` we built above still carry their local placeholder ids.
created += [node for node in reloaded.nodes.values() if node.id.startswith("http")]

[0%] Saving Organization _:000001


[33%] Saving Person _:000000


[66%] Saving Person _:000002


Because these nodes already exist, the upload matched them by their existence queries and updated
them in place rather than creating duplicates — the same machinery as section 4, applied in bulk.
```

## &#9711; 9. Registering files by hand, and why you usually should not

Section 5 described a repository and stopped there, because the Knowledge Graph ingests the files in
the bucket for you. Sometimes, though, you do need `File` nodes of your own: data that are not in a
Collaboratory bucket, or a handful of files you want to describe individually.

Here is how, together with the reason it is the exception.

In [29]:
manual_repository = omcore.FileRepository(
    name=f"{DATASET_SHORT_NAME} manual repository",
    iri=IRI("https://example.org/LH-001-manual"),
    hosted_by=mgm,
)
manual_repository.save(client, space=SPACE, recursive=False)

byte = terms.UnitOfMeasurement.byte     # the attribute form - see the note below

manual_files = [
    omcore.File(
        name=name,
        # An id of our own: see the note below.
        id=client.uri_from_uuid(str(uuid4())),
        iri=IRI(f"{manual_repository.iri.value}/{name}"),
        # by_name() here: the attribute form of a content type is hard to guess
        format=omcore.ContentType.by_name(content_type),
        storage_size=omcore.QuantitativeValue(value=float(size), unit=byte),
        hashes=omcore.Hash(algorithm="SHA1", digest=digest),
        file_repository=manual_repository,
    )
    for name, content_type, size, digest in [
        ("participants.tsv", "text/tab-separated-values", 412,
         "da39a3ee5e6b4b0d3255bfef95601890afd80709"),
        ("recording_001.abf", "application/octet-stream", 647168,
         "5baa61e4c9b93f3f0682250b6cf8331b7ee68fd8"),
    ]
]

for f in manual_files:
    f.save(client, space=SPACE, recursive=False)

created += [manual_repository] + manual_files
print([f.uuid for f in manual_files])

['cec970df-b939-474b-91f7-dfebf4d7515f', '31dcfec8-2d86-4220-8f6d-8e2a3db8742a']


**Why give each file an `id` of our own?** Look at how a `File` is matched:

In [30]:
print(omcore.File.existence_query_properties)

('iri', 'hashes')


Without an id, saving one file asks the Knowledge Graph to search *every* `File` node it holds for a
matching IRI and hash, and there are a great many files. Measured against pre-production while writing
this notebook, a single file took **over 50 seconds** that way, and **under 3 seconds** with an id
supplied; at busy moments the first form did not complete at all. For a dataset with a thousand
files that is the difference between a coffee break and a working day.

So when you know a node is new — which, for files you have just catalogued, you do — generate the id
yourself. You give up the duplicate protection and skip a very expensive query.

And then consider whether you needed to create those nodes at all: for data in a Collaboratory
bucket, you did not.

## &#11088; 10. Cleaning up after yourself

**Please run this before you leave.** Everything above is test data in your private space; there is
no reason to keep it, and a tidy space makes your next experiment easier to read.

In [31]:
print(f"deleting {len(created)} nodes from {SPACE}")

for obj in reversed(created):          # children last, so links do not dangle on the way out
    try:
        obj.delete(client)
    except Exception as err:
        print(f"  could not delete {type(obj).__name__}: {type(err).__name__}")

remaining = client.space_info(SPACE, release_status="in progress")
left = {cls.__name__: n for cls, n in remaining.items() if n and cls in ours}
print("of the types we created, left in", SPACE, ":", left or "nothing")

deleting 14 nodes from myspace


of the types we created, left in myspace : {'DOI': 1, 'DatasetVersion': 1, 'File': 6, 'FileRepository': 1, 'Organization': 2, 'Person': 4}


If a space becomes thoroughly tangled, there is a blunter instrument:

```python
client.clean_space("myspace")      # deletes EVERYTHING in the space
```

It is left commented out deliberately. It cannot be undone, and it is only ever appropriate for a
private space you are using as a scratchpad.

## &#9711; 11. Where to go next

**Moving to production.** Everything here works unchanged against the production KG: pass
`host="core.kg.ebrains.eu"` instead. You will still be writing to your own `myspace` unless you have
been granted write access elsewhere.

**Documentation.**
[Creating and updating metadata](https://fairgraph.readthedocs.io/en/latest/creatingupdating.html)
and [access permissions](https://fairgraph.readthedocs.io/en/latest/permissions.html).

**Other routes.** Most data providers use the **Metadata Wizard** rather than writing metadata
themselves, and never see openMINDS directly. If your data are in BIDS,
[bids2openminds](https://github.com/HumanBrainProject/bids2openminds) generates openMINDS metadata
from the dataset, which you can then upload exactly as in section 8.

**Reading metadata back out** is the subject of the companion notebook,
`data_consumer_tutorial.ipynb`, in this same directory.

**Help.** support@ebrains.eu, and the
[issue tracker](https://github.com/HumanBrainProject/fairgraph/issues).

## Appendix: exercise solutions

In [32]:
# Section 2: what does Organization.digital_identifier want?

# note the plural: the property is "digital_identifiers"
print(omcore.Organization.property_names)
print([t.__name__ for t in omcore.Organization.get_property("digital_identifiers").types])

broken = omcore.Organization(full_name="Pathe Exchange", digital_identifiers="https://ror.org/xxx")
print(broken.validate())

fixed = omcore.Organization(
    full_name="Pathe Exchange",
    digital_identifiers=[omcore.RORID(identifier="https://ror.org/05dxps055")],
)
print(fixed.validate())

['affiliations', 'digital_identifiers', 'full_name', 'has_parents', 'homepage', 'short_name']
['GRIDID', 'RORID', 'RRID']
{'type': ['digital_identifiers: Expected GRIDID, RORID, RRID, value contains str']}
{}


In [33]:
# Section 4: two versions sharing a version_identifier

print(omcore.DatasetVersion.existence_query_properties)

# Matching is on BOTH short_name and version_identifier, so two versions with the same
# version_identifier but different short names are different nodes. Within one dataset, though,
# re-using "v1" for a second version would match the first and overwrite it.

('short_name', 'version_identifier')


In [34]:
# Section 5: a second version, linked to the first

dsv_2 = omcore.DatasetVersion(
    full_name=DATASET_NAME,
    short_name=DATASET_SHORT_NAME,
    version_identifier="v1.1",
    version_innovation="Corrected the timing annotations.",
    release_date=date(1942, 3, 1),
    full_documentation=omcore.WebResource(iri=IRI("https://example.org/LH-001/README.md")),
    is_new_version_of=dsv,
    **controlled,
)
print(dsv_2.validate())

# To attach it to the parent, add it to has_versions and save the parent again:
#   dataset.has_versions = [dsv, dsv_2]
#   dataset.save(client)

{'required': ['digital_identifier is required, but was not provided', 'digital_identifier is required, but was not provided']}
